In [1]:
import gradio as gr
import mlx_whisper
import os
import json

from functools import partial
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai = OpenAI()
ollama = OpenAI(api_key='ollama', base_url="http://localhost:11434/v1")

In [5]:
MODELS = {
    'GPT 5.4 mini': 'gpt-5.4-mini', 
    'GPT 5 mini': 'gpt-5-mini',
    'llama 3.2': 'llama3.2',
}
EVAL_MODELS = MODELS
EVAL_MODELS['no eval'] = 'none'

settings = {}
usage = {'coach_input': 0, 'coach_output': 0, 'coach_total': 0, 'coach_$': 0,
         'ev_input': 0, 'ev_output': 0, 'ev_total': 0, 'ev_$': 0
        }

settings_file = os.path.join('..', 'settings.json')

def set_sys_prompts():  
    with open(settings_file, 'r') as f:
        settings_obj = json.load(f)
        
    settings['COACH_PROMPT'] = settings_obj.get('COACH_PROMPT', 
                                                settings_obj.get('DEFAULT_COACH_PROMPT'))
    settings['ENGLISH_PROMPT'] = settings_obj.get('ENGLISH_PROMPT',
                                                 settings_obj.get('DEFAULT_ENGLISH_PROMPT'))


In [6]:
def ask_openai(client, model, message):
    messages = [{"role": 'user', "content": f"{settings['ENGLISH_PROMPT']}{message}"}]
    completion = client.chat.completions.create(
        model=model,
        messages=messages
    )

    u = completion.usage
    usage['ev_input'] += u.prompt_tokens
    usage['ev_output'] += u.completion_tokens
    usage['ev_total'] += u.total_tokens
    usage_text = (f"inputs + {u.prompt_tokens}: {usage['ev_input']:,} | "
                  f"output + {u.completion_tokens}: {usage['ev_output']:,} | "
                  f"total + {u.total_tokens}: {usage['ev_total']:,}")
    
    return completion.choices[0].message.content, usage_text
    

def stream_openai(client, model, message, history):
    messages = [{"role": "system", "content": settings['COACH_PROMPT']}]
    

    for record in history:
        messages.append({"role": record["role"], "content": record["content"][0]["text"]})
        
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
        stream_options={"include_usage": True}
    )

    partial_reply = ""
    history_chunk = []
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            partial_reply += chunk.choices[0].delta.content
            history_chunk = [{"role": "user", "content": [{"text": message, "type": "text"}]},
                            {"role": "assistant", "content": [{"text": partial_reply, "type": "text"}]}]
            
            yield history + history_chunk, gr.update()
        elif chunk.usage is not None:
            u = chunk.usage
            usage['coach_input'] += u.prompt_tokens
            usage['coach_output'] += u.completion_tokens
            usage['coach_total'] += u.total_tokens
            usage_text = (f"inputs + {u.prompt_tokens}: {usage['coach_input']:,} | "
                          f"output + {u.completion_tokens}: {usage['coach_output']:,} | "
                          f"total + {u.total_tokens}: {usage['coach_total']:,}")
            yield gr.update(), usage_text


def msg_submit(message, history, model='GPT 5 mini'):
    if message:
        model_name = MODELS[model]
        if model_name.startswith('gpt'):
            yield from stream_openai(openai, model_name, message, history)
        elif model_name.startswith('llama'):
            yield from stream_openai(ollama, model_name, message, history)
        else:
            error = f"Error: unknown model name: {model_name}"
            print(error)
            gr.Error(error)
    else:
        yield history, gr.update()


def msg_change(message):
    return gr.update(interactive=bool(message.strip()))


def transcribe(msg, audio):
    text = msg
    if audio:
        try:
            result = mlx_whisper.transcribe(
                audio,
                path_or_hf_repo="mlx-community/whisper-small-mlx"
            )
        
            text += result["text"]
        except Exception as e:
            gr.Warning(f"Transcription failed: {e}")
    return text, None


def sysprompt_update(prompt, key='COACH_PROMPT'):
    if key in settings.keys():
        settings[key] = prompt
        with open(settings_file, 'r') as f:
            settings_obj = json.loads(f.read())
        
        settings_obj[key] = prompt
        with open(settings_file, 'w') as f:
            f.write(json.dumps(settings_obj, ensure_ascii=False))

        if key == 'COACH_PROMPT':
            gr.Info("Coach system prompt updated.")
        elif key == 'ENGLISH_PROMPT':
            gr.Info("English evaluation system prompt updated.")
            
    else:
        gr.Warning(f'Failed to update: unknown setting {key}.')
        return gr.update()
        
    return prompt


def evaluate(message, history, model):
    if not message or model == 'no eval':
        print(f'Skip evaluation...')
        return "", history, gr.update()
        
    model_name = MODELS[model]
    if model_name.startswith('gpt'):
        model_resp, usage_text = ask_openai(openai, model_name, message)
    elif model_name.startswith('llama'):
        model_resp, usage_text = ask_openai(ollama, model_name, message)
    else:
        error = f"Error: unknown model name: {model_name}"
        print(error)
        gr.Error(error)
        return "", history, gr.update()
        
    evaluation = f"Message: `{message}`\n{model_resp}"
    new_text = f"{evaluation}\n---\n{history}" if history else f"{evaluation}"

    return "", new_text, usage_text


def format_usage(key='coach'):
    def get(field):
        return usage[f"{key}_{field}"]

    return (
        f"input {get('input'):,} | "
        f"output {get('output'):,} | "
        f"total {get('total'):,}"
    )
    
def render_usages():
    return format_usage(), format_usage(key='ev')


def render_sys_prompts():
    coach_prompt = settings['COACH_PROMPT']
    ev_prompt = settings['ENGLISH_PROMPT']

    return coach_prompt, ev_prompt


def clear_coach_usage():
    usage["coach_input"] = 0
    usage["coach_output"] = 0
    usage["coach_total"] = 0
    usage["coach_$"] = 0

    return format_usage()

def clear_ev_usage():
    usage["ev_input"] = 0
    usage["ev_output"] = 0
    usage["ev_total"] = 0
    usage["ev_$"] = 0

    return format_usage(key='ev')

In [7]:
css = """
.gr_btn{
    max-width: 80px;
    min-width: 40px;
}

.chatbot {
    height: 60vh !important;
}
"""

set_sys_prompts()

with gr.Blocks() as ui:
    with gr.Tabs():
        with gr.Tab("Interview coach"):
            with gr.Row():
                with gr.Sidebar(open=False):
                    gr.Markdown("### Settings")
                    sysprompt_inp = gr.Textbox(label='System Prompt')
                    sysprompt_update_btn = gr.Button(value='Update')
                with gr.Column():
                    with gr.Row():
                        coach_usage = gr.Markdown()
                        clear_coach_usage_btn = gr.Button(value='Clear', elem_classes='gr_btn')
                    chatbot = gr.Chatbot(layout='panel', elem_classes='chatbot')
                    with gr.Row():
                        audio_msg = gr.Audio(sources="microphone", type="filepath",
                                        container=True, recording=False,
                                        waveform_options=gr.WaveformOptions(
                                            show_recording_waveform=False,
                                        ), scale=2)
                        with gr.Column(scale=1, min_width=80):
                            coach_model = gr.Dropdown(choices=MODELS, value='GPT 5.4 mini', show_label=False, container=False,
                                                interactive=True)
                            eval_model = gr.Dropdown(choices=EVAL_MODELS, value='llama 3.2', show_label=False, container=False,
                                                interactive=True)
                        clear_btn = gr.Button(value='Clear', elem_classes='gr_btn')
                    with gr.Row():
                        send_btn = gr.Button(value='Send', interactive=False, elem_classes='gr_btn')   
                        msg = gr.Textbox(elem_id='msg', show_label=False, container=False)

        with gr.Tab("English evaluation"):
            with gr.Row():
                with gr.Sidebar(open=False):
                    gr.Markdown("### Settings")
                    ev_sysprompt_inp = gr.Textbox(label='System Prompt')
                    ev_sysprompt_update_btn = gr.Button(value='Update')
                with gr.Column():
                    with gr.Row():
                        ev_usage = gr.Markdown()
                        clear_ev_usage_btn = gr.Button(value='Clear', elem_classes='gr_btn')
                    ev_mrk = gr.Markdown('')

    ui.load(render_usages, outputs=[coach_usage, ev_usage])
    ui.load(render_sys_prompts, outputs=[sysprompt_inp, ev_sysprompt_inp])
    
    msg.change(msg_change, [msg], [send_btn])
    msg.submit(msg_submit, [msg, chatbot, coach_model], [chatbot, coach_usage]
              ).then(evaluate, [msg, ev_mrk, eval_model], [msg, ev_mrk, ev_usage])
    
    send_btn.click(msg_submit, [msg, chatbot, coach_model], [chatbot, coach_usage]
              ).then(evaluate, [msg, ev_mrk, eval_model], [msg, ev_mrk, ev_usage])

    audio_msg.input(transcribe, inputs=[msg, audio_msg], outputs=[msg, audio_msg]
                   ).then(fn=None, js="() => document.querySelector('#msg textarea')?.focus()")


    sysprompt_inp.submit(fn=sysprompt_update, inputs=[sysprompt_inp], outputs=sysprompt_inp)
    sysprompt_update_btn.click(fn=sysprompt_update, inputs=[sysprompt_inp], outputs=sysprompt_inp)

    ev_sysprompt_inp.submit(fn=partial(sysprompt_update, key='ENGLISH_PROMPT'), inputs=[ev_sysprompt_inp], outputs=None)
    ev_sysprompt_update_btn.click(fn=partial(sysprompt_update, key='ENGLISH_PROMPT'), inputs=[ev_sysprompt_inp], outputs=None)

    clear_btn.click(lambda: None, inputs=None, outputs=chatbot, queue=False)
    clear_coach_usage_btn.click(clear_coach_usage, inputs=None, outputs=coach_usage, queue=False)
    clear_ev_usage_btn.click(clear_ev_usage, inputs=None, outputs=ev_usage, queue=False)
    
ui.launch(inbrowser=False, css=css)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]